# 拟合优度检验完整教程：卡方检验的原理与应用

## 📚 教学目标
1. 理解卡方拟合优度检验的原理和适用场景
2. 掌握观测频数和期望频数的计算方法
3. 手算卡方统计量并理解其含义
4. 用 scipy.stats.chisquare 验证手算结果
5. 了解本福特定律的实际应用

**参考书**：《基础统计学(第14版)》（Triola）第 11-1 节
**教学策略**：从掷骰子的简单例子入手，逐步理解拟合优度检验的完整流程

---

## 1. 场景设定：骰子是否公平？

### 🎯 问题

赌场怀疑一个骰子可能被动了手脚。我们掷了 60 次，记录每个面出现的次数：

| 面 | 1 | 2 | 3 | 4 | 5 | 6 |
|----|---|---|---|---|---|---|
| 观测频数 | 8 | 6 | 12 | 15 | 11 | 8 |

如果骰子是公平的，每个面应该有 10 次（60/6 = 10）。
现在的问题是：观测到的频数与期望频数的差异是否足以说明骰子不公平？

### 📖 书中的定义

> 拟合优度检验（goodness-of-fit test）用于判断观测频数分布是否与某个理论分布一致。
> 零假设 H₀：观测频数分布与期望频数分布一致。
> 备择假设 H₁：观测频数分布与期望频数分布不一致。

### 📐 卡方统计量

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

其中：
- O = 观测频数 (Observed)
- E = 期望频数 (Expected)
- 求和遍及所有类别

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据手算：掷骰子

### 📐 计算步骤

1. 确定观测频数 O
2. 计算期望频数 E
3. 计算每个类别的 $(O - E)^2 / E$
4. 求和得到卡方统计量
5. 确定自由度 df = k - 1
6. 查表或计算 p 值

In [ ]:
# ========== 步骤 1: 手算卡方检验 ==========

print("=" * 60)
print("📋 拟合优度检验: 骰子是否公平？")
print("=" * 60)

# 数据
observed = np.array([8, 6, 12, 15, 11, 8])
n_total = np.sum(observed)
k = len(observed)  # 类别数
expected = np.array([n_total / k] * k)  # 公平骰子的期望频数

print(f"\n📊 步骤 1: 数据")
print(f"  总次数 n = {n_total}")
print(f"  类别数 k = {k}")
print(f"  观测频数 O = {observed}")
print(f"  期望频数 E = {expected}")

# 手算卡方统计量
chi2_components = (observed - expected)**2 / expected
chi2_manual = np.sum(chi2_components)

print(f"\n📊 步骤 2: 计算卡方统计量")
print(f"{'面':<6} {'O':<8} {'E':<8} {'O-E':<8} {'(O-E)²':<10} {'(O-E)²/E':<10}")
print("-" * 50)
for i in range(k):
    print(f"  {i+1:<6} {observed[i]:<8} {expected[i]:<8.1f} {observed[i]-expected[i]:<8.1f} {(observed[i]-expected[i])**2:<10.2f} {chi2_components[i]:<10.4f}")
print(f"  {'合计':<6} {n_total:<8} {n_total:<8.1f} {'':8} {'':10} {chi2_manual:<10.4f}")

# 自由度
df = k - 1
print(f"\n📊 步骤 3: 自由度")
print(f"  df = k - 1 = {k} - 1 = {df}")

# p 值
p_value = 1 - stats.chi2.cdf(chi2_manual, df)
print(f"\n📊 步骤 4: p 值")
print(f"  χ² = {chi2_manual:.4f}")
print(f"  p 值 = {p_value:.4f}")

# 判断
alpha = 0.05
print(f"\n📊 步骤 5: 判断 (α = {alpha})")
if p_value < alpha:
    print(f"  p = {p_value:.4f} < {alpha} → 拒绝 H₀")
    print(f"  💡 有足够的证据说明骰子不公平")
else:
    print(f"  p = {p_value:.4f} ≥ {alpha} → 不能拒绝 H₀")
    print(f"  💡 没有足够的证据说明骰子不公平")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 2: scipy 验证 ==========

print("=" * 60)
print("🔬 scipy.stats.chisquare 验证")
print("=" * 60)

chi2_scipy, p_scipy = stats.chisquare(observed, expected)

print(f"\n📊 手算 vs scipy 对比:")
print(f"  手算 χ² = {chi2_manual:.6f}")
print(f"  scipy χ² = {chi2_scipy:.6f}")
print(f"  手算 p = {p_value:.6f}")
print(f"  scipy p = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 观测 vs 期望 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图对比
ax1 = axes[0]
x_pos = np.arange(k)
width = 0.35
bars1 = ax1.bar(x_pos - width/2, observed, width, label='Observed', color='steelblue', alpha=0.8)
bars2 = ax1.bar(x_pos + width/2, expected, width, label='Expected', color='#e74c3c', alpha=0.8)
ax1.set_xlabel('Die Face', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Observed vs Expected Frequency', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(['1', '2', '3', '4', '5', '6'])
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 卡方分布 + 临界值
ax2 = axes[1]
x_chi2 = np.linspace(0, 20, 1000)
y_chi2 = stats.chi2.pdf(x_chi2, df)
ax2.plot(x_chi2, y_chi2, 'b-', linewidth=2, label=f'χ² Distribution (df={df})')
ax2.fill_between(x_chi2, 0, y_chi2, alpha=0.1, color='steelblue')

# 临界值
chi2_critical = stats.chi2.ppf(1 - alpha, df)
x_reject = np.linspace(chi2_critical, 20, 100)
y_reject = stats.chi2.pdf(x_reject, df)
ax2.fill_between(x_reject, 0, y_reject, alpha=0.3, color='#e74c3c', label='Rejection Region')

# 标记检验统计量
ax2.axvline(x=chi2_manual, color='#e67e22', linestyle='--', linewidth=2, label=f'χ² = {chi2_manual:.2f}')
ax2.axvline(x=chi2_critical, color='red', linestyle=':', linewidth=1.5, label=f'Critical Value = {chi2_critical:.2f}')

ax2.set_xlabel('χ² Value', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('Chi-Square Distribution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：观测频数与期望频数的柱状图对比")
print("  右图：卡方分布，橙色虚线为检验统计量，红色虚线为临界值")
print(f"  如果 χ² = {chi2_manual:.2f} 落入红色拒绝区域，则拒绝 H₀")

---

## 3. 大样本示例：血型分布

### 🎯 问题

某医院统计了 500 名患者的血型分布，已知该地区血型的理论分布为：
- A 型: 40%
- B 型: 30%
- O 型: 20%
- AB 型: 10%

观测到的分布为：A=190, B=150, O=110, AB=50

检验：该医院患者的血型分布是否与该地区一致？

In [ ]:
# ========== 步骤 3: 大样本拟合优度检验 ==========

print("=" * 60)
print("📋 拟合优度检验: 血型分布")
print("=" * 60)

# 数据
observed_blood = np.array([190, 150, 110, 50])
proportions = np.array([0.40, 0.30, 0.20, 0.10])
n_total_blood = np.sum(observed_blood)
expected_blood = n_total_blood * proportions

print(f"\n📊 数据:")
print(f"  总人数 n = {n_total_blood}")
print(f"{'血型':<6} {'理论比例':<10} {'观测频数':<10} {'期望频数':<10}")
print("-" * 36)
blood_types = ['A', 'B', 'O', 'AB']
for i in range(len(blood_types)):
    print(f"  {blood_types[i]:<6} {proportions[i]:<10.2f} {observed_blood[i]:<10} {expected_blood[i]:<10.1f}")

# 卡方检验
chi2_blood, p_blood = stats.chisquare(observed_blood, expected_blood)
df_blood = len(observed_blood) - 1

print(f"\n📊 检验结果:")
print(f"  χ² = {chi2_blood:.4f}")
print(f"  df = {df_blood}")
print(f"  p 值 = {p_blood:.6f}")

alpha = 0.05
if p_blood < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：该医院患者的血型分布与该地区不一致")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀：没有足够证据说明分布不一致")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 血型分布 ==========

fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(blood_types))
width = 0.35
bars1 = ax.bar(x_pos - width/2, observed_blood, width, label='Observed', color='steelblue', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, expected_blood, width, label='Expected', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Blood Type', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Blood Type Distribution: Observed vs Expected', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(blood_types)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# 在柱子上标注数值
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  蓝色：观测频数，红色：期望频数")
print(f"  χ² = {chi2_blood:.4f}, p = {p_blood:.4f}")
print("  差异越大，卡方值越大，越可能拒绝 H₀")

---

## 4. 模拟验证：卡方统计量的分布

### 💡 核心思想

如果 H₀ 为真（骰子公平），多次重复实验得到的卡方统计量应该服从卡方分布。
我们用模拟来验证这一点。

In [ ]:
# ========== 步骤 4: 模拟验证 ==========

print("=" * 60)
print("📋 模拟验证: 卡方统计量的抽样分布")
print("=" * 60)

# 模拟参数
n_sims = 10000
n_rolls = 60  # 每次掷骰子次数
k_faces = 6   # 骰子面数

# 存储每次模拟的卡方值
chi2_simulated = []

for _ in range(n_sims):
    # 模拟公平骰子
    rolls = np.random.randint(0, k_faces, n_rolls)
    obs = np.bincount(rolls, minlength=k_faces)
    exp = np.array([n_rolls / k_faces] * k_faces)
    chi2_val = np.sum((obs - exp)**2 / exp)
    chi2_simulated.append(chi2_val)

chi2_simulated = np.array(chi2_simulated)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))

# 模拟分布
ax.hist(chi2_simulated, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white', label='Simulated χ²')

# 理论卡方分布
x_chi2 = np.linspace(0, 25, 1000)
y_chi2 = stats.chi2.pdf(x_chi2, df=k_faces-1)
ax.plot(x_chi2, y_chi2, 'r-', linewidth=2, label=f'Theoretical χ² (df={k_faces-1})')

# 标记我们的观测值
ax.axvline(x=chi2_manual, color='#e67e22', linestyle='--', linewidth=2, label=f'Our χ² = {chi2_manual:.2f}')

ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Simulated vs Theoretical Chi-Square Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 图解说明：")
print(f"  蓝色直方图：{n_sims} 次模拟的卡方统计量分布")
print(f"  红色曲线：理论卡方分布 (df={k_faces-1})")
print(f"  橙色虚线：我们观测到的 χ² = {chi2_manual:.2f}")
print(f"  模拟分布与理论分布高度吻合，验证了卡方检验的理论基础")

---

## 5. 本福特定律

### 📖 书中的要点

> 本福特定律（Benford's Law）描述了在许多自然数据集中，首位数字的分布规律。
> 数字 1 作为首位数字的概率约为 30.1%，而数字 9 作为首位数字的概率仅为 4.6%。

### 📐 本福特定律公式

$$P(d) = \log_{10}\left(1 + \frac{1}{d}\right), \quad d = 1, 2, \ldots, 9$$

### 💡 应用场景

- 财务报表欺诈检测
- 选举数据异常检测
- 科学数据造假识别

In [ ]:
# ========== 步骤 5: 本福特定律 ==========

print("=" * 60)
print("📋 本福特定律: 首位数字分布")
print("=" * 60)

# 本福特定律的理论概率
digits = np.arange(1, 10)
benford_prob = np.log10(1 + 1/digits)

print(f"\n📊 本福特定律理论概率:")
print(f"{'首位数字':<10} {'理论概率':<10}")
print("-" * 20)
for d, p in zip(digits, benford_prob):
    print(f"  {d:<10} {p:<10.4f}")

# 模拟: 生成符合本福特定律的数据
n_data = 1000
simulated_data = []
for _ in range(n_data):
    # 生成随机数，取首位数字
    num = np.random.exponential(1000)
    num_str = str(int(num))
    first_digit = int(num_str.lstrip('0')[0]) if num_str.lstrip('0') else 1
    simulated_data.append(first_digit)

simulated_data = np.array(simulated_data)
observed_benford = np.bincount(simulated_data, minlength=10)[1:]  # 忽略 0
expected_benford = n_data * benford_prob

# 卡方检验
chi2_benford, p_benford = stats.chisquare(observed_benford, expected_benford)

print(f"\n📊 模拟数据检验:")
print(f"  样本量 n = {n_data}")
print(f"{'首位数字':<10} {'观测频数':<10} {'期望频数':<10}")
print("-" * 30)
for i in range(9):
    print(f"  {digits[i]:<10} {observed_benford[i]:<10} {expected_benford[i]:<10.1f}")

print(f"\n📊 卡方检验结果:")
print(f"  χ² = {chi2_benford:.4f}")
print(f"  df = 8")
print(f"  p 值 = {p_benford:.4f}")

alpha = 0.05
if p_benford < alpha:
    print(f"  💡 p < {alpha}，拒绝 H₀：数据不符合本福特定律")
else:
    print(f"  💡 p ≥ {alpha}，不能拒绝 H₀：数据可能符合本福特定律")

In [ ]:
# ========== 可视化: 本福特定律 ==========

fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(digits))
width = 0.35
bars1 = ax.bar(x_pos - width/2, observed_benford, width, label='Observed', color='steelblue', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, expected_benford, width, label='Benford Expected', color='#e74c3c', alpha=0.8)

ax.set_xlabel('First Digit', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title("Benford's Law: First Digit Distribution", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(digits)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  蓝色：模拟数据的首位数字分布")
print("  红色：本福特定律的理论分布")
print("  许多自然数据集的首位数字遵循本福特定律")
print("  如果数据严重偏离本福特定律，可能存在人为操纵")

---

## 📌 核心概念回顾

### 📌 拟合优度检验 (Goodness-of-Fit Test)
- **定义**: 判断观测频数分布是否与某个理论分布一致
- **公式**: $\chi^2 = \sum \frac{(O - E)^2}{E}$
- **含义**: 卡方值越大，观测与期望的差异越大
- **判断标准**: p < α 时拒绝 H₀

### 📌 自由度 (Degrees of Freedom)
- **定义**: 类别数减去约束条件数
- **公式**: df = k - 1（无参数估计时）
- **含义**: 决定卡方分布的形态

### 📌 期望频数 (Expected Frequency)
- **定义**: 在 H₀ 为真的假设下，每个类别的理论频数
- **公式**: E = n × p（总次数 × 理论概率）
- **要求**: 每个类别的期望频数应 ≥ 5

### 📌 本福特定律 (Benford's Law)
- **定义**: 自然数据集中首位数字的分布规律
- **公式**: $P(d) = \log_{10}(1 + 1/d)$
- **应用**: 欺诈检测、数据质量检验

### 🔗 完整流程
```
设定假设 → 计算期望频数 → 计算 χ² → 确定 df → 求 p 值 → 判断
    ↓            ↓           ↓        ↓        ↓       ↓
  H₀/H₁      E = n×p    Σ(O-E)²/E   k-1    查表/软件   p<α?
```

---

## ❌ 常见误区

### ❌ 误区 1: 期望频数太小仍使用卡方检验
**✓ 正确做法**: 每个类别的期望频数应至少为 5。如果某些类别的期望频数 < 5，应合并相邻类别或使用 Fisher 精确检验。

### ❌ 误区 2: 拟合优度检验可以证明 H₀ 为真
**✓ 正确理解**: 不能拒绝 H₀ 只表示没有足够证据说明分布不一致，不等于证明 H₀ 为真。可能是样本量太小，检验功效不足。

### ❌ 误区 3: 卡方值大就一定是显著的
**✓ 正确理解**: 卡方值是否显著取决于自由度和样本量。同样的卡方值，在不同自由度下可能有不同的 p 值。应结合 p 值和效应量综合判断。

### ❌ 误区 4: 混淆拟合优度检验和独立性检验
**✓ 正确理解**: 拟合优度检验用于判断单个分类变量的分布是否与理论分布一致；独立性检验用于判断两个分类变量是否独立。两者的卡方统计量公式相同，但假设和解释不同。

### ❌ 误区 5: 忽略本福特定律的适用条件
**✓ 正确理解**: 本福特定律适用于跨越多个数量级的自然数据集。对于有上限或下限约束的数据（如身高、血压），本福特定律不适用。